# octlm on Colab

Runs the training experiments on a Colab GPU. Set **Runtime > Change runtime type > GPU**
before anything else, then run the cells in order.

This notebook uses Colab's preinstalled PyTorch rather than `uv sync`, because the lockfile
pins the CPU build. The Python and PyTorch versions therefore differ from the local
environment. Timings from Colab are not comparable with the CPU timings in `notes/`; loss
and bits per byte are, as long as the corpus is the same one (see the corpus cell).

In [ ]:
!nvidia-smi
import torch

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Repository

Private repo: put a token in `TOKEN` and the clone URL becomes
`https://{TOKEN}@github.com/whynotramaa/octlm.git`.

In [ ]:
import os
import pathlib

REPO = "https://github.com/whynotramaa/octlm.git"
BRANCH = "main"
if pathlib.Path("/content/octlm/.git").exists():
    !cd /content/octlm && git fetch origin && git checkout $BRANCH && git pull
else:
    !git clone --branch $BRANCH $REPO /content/octlm
os.chdir("/content/octlm")
!git log --oneline -1

## Corpus

`octlm.corpus` reads the Python standard library of the machine it runs on, so this build
belongs to Colab's Python image. That build is the canonical Day 2 corpus. Check the
fingerprints against `notes/day2.md` before comparing a run against the recorded numbers,
because a Colab image upgrade changes the corpus.

The VM is deleted between sessions. Copy `data/` and `artifacts/day2/` to Drive with the
next cell to skip the 5-minute tokenizer and block build next time.

In [ ]:
!python -m octlm.corpus
!ls -la data artifacts/day2 2>/dev/null

In [ ]:
# Optional: keep the corpus and the block cache across sessions.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/octlm
# !cp -r data artifacts /content/drive/MyDrive/octlm/     # save
# !cp -r /content/drive/MyDrive/octlm/data /content/drive/MyDrive/octlm/artifacts .  # restore

## Smoke check

Builds the model and prints the config hash, parameter count, and logits shape.

In [ ]:
!python -m octlm.train --config configs/day2.toml --dry-run
!python -m octlm.bench --dummy --contexts 128 512 2048 --device cuda

## Day 2 training runs

`variants` is the architecture grid, `length` is the context sweep. These are the two stages
worth a GPU. `tiled`, `equivalence`, `sdpa`, and `cache` are CPU measurements that are already
recorded in `notes/day2.md`; the SDPA benchmark measures process RSS, which means nothing on a
GPU, so leave those on CPU.

In [ ]:
!python -m octlm.day2 variants --device cuda

In [ ]:
!python -m octlm.day2 length --config configs/day2-long.toml --device cuda

In [ ]:
!python -m octlm.day2 report

## Day 1 trainer

`--device auto` is the default everywhere, so `--device cuda` is only needed to be explicit or
to force CPU for a comparison.

In [ ]:
!python -m octlm.train --config configs/day2.toml --tokenizer bpe \
    --train data/train.jsonl --validation data/val.jsonl \
    --checkpoint artifacts/day2/colab.pt --metrics runs/colab-day2.jsonl --device cuda

## Day 3a runs

Stage 0 first. It measures the seed spread at the Day 3 budget and decides whether the rest of the
day is readable. Do not run the experiments until you have looked at its report.

Each stage takes minutes and a free Colab session drops, so run them one cell at a time and copy
`runs/` out before the VM dies. A cell driven over the MCP bridge times out at 30 seconds, so the
long stages go through `subprocess.Popen` into a log file.

In [ ]:
import pathlib
import subprocess

pathlib.Path("logs").mkdir(exist_ok=True)


def run(name, command):
    """Detach the stage into a log file so a slow cell cannot hit the 30-second tool timeout."""
    log = open(f"logs/{name}.log", "w")
    stage = ["python", "-m", *command.split()]
    return subprocess.Popen(stage, stdout=log, stderr=subprocess.STDOUT)


def tail(name, lines=15):
    print("".join(open(f"logs/{name}.log").readlines()[-lines:]))

### Stage 0: the noise floor

Day 2's grid moved 0.043 bits per byte across three seeds with nothing changed. This reruns
`baseline`, `swiglu`, and `modern` at 2000 steps instead of 400, on the same width and the same
data, and writes to its own file so a 2000-step record never mixes with a 400-step one.

Read the spread before anything else. Above 0.03, raise `steps` in `configs/day3.toml` to 4000 and
rerun this cell. It also gives the Day 2 open question, `modern` against `swiglu`, its cheapest
shot.

In [ ]:
floor = run(
    "floor",
    "octlm.day2 variants --config configs/day3.toml --variants baseline swiglu modern"
    " --out runs/day3-floor.jsonl --device cuda",
)
floor.wait()
tail("floor")

In [ ]:
!python -m octlm.day2 report --out runs/day3-floor.jsonl

### EXP-016: multi-token prediction

Depths 1, 2, and 3. Depth 1 must reproduce the pre-Day-3 model exactly. Watch
`depth2_top1_agreement`: that is the number Phase 5 needs to decide whether a self-draft head is
worth building for speculative decoding.

In [ ]:
mtp = run("mtp", "octlm.day3 mtp --config configs/day3.toml --device cuda")
mtp.wait()
tail("mtp")

### EXP-017 and EXP-018: sparse and compressed attention

Both run at context 512 (`configs/day3-long.toml`), because a 128-token window means nothing
against a 256-token context. The copy probe rides along on the first seed of each, so it costs no
extra training.

`flash_accepts_mask` is measured on the training device on purpose. PyTorch's CPU flash path takes
an explicit mask and the CUDA kernel does not, so a CPU probe would report the wrong backend for a
CUDA run.

In [ ]:
sparse = run("sparse", "octlm.day3 sparse --config configs/day3-long.toml --device cuda")
sparse.wait()
tail("sparse", 25)

In [ ]:
compressed = run(
    "compressed", "octlm.day3 compressed --config configs/day3-long.toml --device cuda"
)
compressed.wait()
tail("compressed", 25)

### EXP-019: MLA

Ranks 32, 64, and 128 against the GQA-2 control. The hypothesis on record is that MLA loses here:
GQA-2 already holds 128 cache dimensions per token per layer, and MLA only undercuts that below
rank 112. The useful output is the crossover, not a winner.

In [ ]:
mla = run("mla", "octlm.day3 mla --config configs/day3-long.toml --device cuda")
mla.wait()
tail("mla", 20)

### Summarize and keep

`report` prints the mean and spread per variant for every stage that has records, then the copy
probe rows. `runs/` and `artifacts/` are gitignored, so copy them out before the VM is deleted.

In [ ]:
!python -m octlm.day3 report

## Keep the results

Colab deletes the VM. `runs/` and `artifacts/` are gitignored, so copy them out.

In [ ]:
!tar czf /content/octlm-results.tar.gz runs artifacts
from google.colab import files

files.download("/content/octlm-results.tar.gz")